#### [HMDA dataset](https://ffiec.cfpb.gov/data-browser/data/2022?category=counties&items=47037)

In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import statsmodels.formula.api as smf
import statsmodels.api as sm

##### Read in the data

In [2]:
# read in the data
hmda_2022 = pd.read_csv('../data/msamd_12060_2022.csv') 
hmda_2023 = pd.read_csv('../data/msamd_12060_2023.csv')

C:\Users\Cat\AppData\Local\Temp\ipykernel_11412\2718690324.py:2: DtypeWarning: Columns (22,23,24,26,27,28,29,30,31,32,33,38,43,44) have mixed types. Specify dtype option on import or set low_memory=False.
  hmda_2022 = pd.read_csv('../data/msamd_12060_2022.csv')
C:\Users\Cat\AppData\Local\Temp\ipykernel_11412\2718690324.py:3: DtypeWarning: Columns (22,23,24,26,27,28,29,30,31,32,33,38,43,44) have mixed types. Specify dtype option on import or set low_memory=False.
  hmda_2023 = pd.read_csv('../data/msamd_12060_2023.csv')


In [3]:
hmda_2022.head(2)

,activity_year,lei,derived_msa-md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,...,denial_reason-2,denial_reason-3,denial_reason-4,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
0,2022,549300FGXN1K3HLB1R50,12060,GA,13135.0,1.313505e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Ethnicity Not Available,...,NaN,NaN,NaN,4582,90.27,95700,59.25,693,1364,26
1,2022,549300FGXN1K3HLB1R50,12060,GA,13113.0,1.311314e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,5777,67.49,95700,117.88,1806,2070,29


In [4]:
len(hmda_2022.columns)

99

In [5]:
# https://stackoverflow.com
# (== number of non-NaN values in first column)
hmda_2022[hmda_2022.columns[0]].count()

np.int64(378284)

### Remove all values that do not reflect approved, denied, or approved but not accepted

In [6]:
# remove all values that do not reflect approved, denied, or approved but not accepted
# Description: The action taken on the covered loan or application
# Values:
# 1 - Loan originated
# 2 - Application approved but not accepted
# 3 - Application denied
# 4 - Application withdrawn by applicant
# 5 - File closed for incompleteness
# 6 - Purchased loan
# 7 - Preapproval request denied
# 8 - Preapproval request approved but not accepted

list_of_values_to_keep = [1,2,3]

hmda_2022_action_taken = hmda_2022[hmda_2022['action_taken'].isin(list_of_values_to_keep)]
hmda_2022_action_taken['action_taken'].unique()

array([1, 2, 3])

In [7]:
# (== number of non-NaN values in first column)
hmda_2022_action_taken[hmda_2022_action_taken.columns[0]].count()

np.int64(254012)

##### Remove not applicable (could include applicants with little to no credit and exempt institutions) to eliminate legal entities like corporations etc from being included in the data instead of individuals

In [8]:
# remove not applicable (could include applicants with little to no credit) and exempt institutions
# to eliminate legal entitities like corporations etc from being included in the data instead of individuals

# Description: The name and version of the credit scoring model used to generate the credit score, or scores, relied on in making the credit decision
# Values:
# 1 - Equifax Beacon 5.0
# 2 - Experian Fair Isaac
# 3 - FICO Risk Score Classic 04
# 4 - FICO Risk Score Classic 98
# 5 - VantageScore 2.0
# 6 - VantageScore 3.0
# 7 - More than one credit scoring model
# 8 - Other credit scoring model
# 9 - Not applicable
# 1111 - Exempt

list_of_values_to_keep = [1,2,3,4,5,6,7,8]

hmda_2022_credit_score_app = hmda_2022_action_taken[hmda_2022_action_taken['applicant_credit_score_type'].isin(list_of_values_to_keep)]
hmda_2022_credit_score_app['applicant_credit_score_type'].unique()

array([2, 1, 3, 8, 7, 6, 4, 5])

In [9]:
# (== number of non-NaN values in first column)
hmda_2022_credit_score_app[hmda_2022_credit_score_app.columns[0]].count()

np.int64(222254)

In [10]:
# remove not applicable (could include applicants with little to no credit) and exempt institutions
# to eliminate legal entitities like corporations etc from being included in the data instead of individuals

# Description: The name and version of the credit scoring model used to generate the credit score, or scores, relied on in making the credit decision
# Values:
# 1 - Equifax Beacon 5.0
# 2 - Experian Fair Isaac
# 3 - FICO Risk Score Classic 04
# 4 - FICO Risk Score Classic 98
# 5 - VantageScore 2.0
# 6 - VantageScore 3.0
# 7 - More than one credit scoring model
# 8 - Other credit scoring model
# 9 - Not applicable
# 10 - No co-applicant
# 11 – FICO Score 9
# 12 – FICO Score 8
# 13 – FICO Score 10
# 14 – FICO Score 10T
# 15 - VantageScore 4.0
# 1111 - Exempt

list_of_values_to_keep = [1,2,3,4,5,6,7,8,10,11,12,13,14,15]

hmda_2022_credit_score_co = hmda_2022_credit_score_app[hmda_2022_credit_score_app['co-applicant_credit_score_type'].isin(list_of_values_to_keep)]
hmda_2022_credit_score_co['co-applicant_credit_score_type'].unique()

array([10,  2,  3,  1,  8,  6,  7,  4,  5, 11])

In [11]:
# (== number of non-NaN values in first column)
hmda_2022_credit_score_co[hmda_2022_credit_score_co.columns[0]].count()

np.int64(181011)

##### Inspect all columns

In [12]:
hmda_2022_credit_score_co.columns

Index(['activity_year', 'lei', 'derived_msa-md', 'state_code', 'county_code',
       'census_tract', 'conforming_loan_limit', 'derived_loan_product_type',
       'derived_dwelling_category', 'derived_ethnicity', 'derived_race',
       'derived_sex', 'action_taken', 'purchaser_type', 'preapproval',
       'loan_type', 'loan_purpose', 'lien_status', 'reverse_mortgage',
       'open-end_line_of_credit', 'business_or_commercial_purpose',
       'loan_amount', 'loan_to_value_ratio', 'interest_rate', 'rate_spread',
       'hoepa_status', 'total_loan_costs', 'total_points_and_fees',
       'origination_charges', 'discount_points', 'lender_credits', 'loan_term',
       'prepayment_penalty_term', 'intro_rate_period', 'negative_amortization',
       'interest_only_payment', 'balloon_payment',
       'other_nonamortizing_features', 'property_value', 'construction_method',
       'occupancy_type', 'manufactured_home_secured_property_type',
       'manufactured_home_land_property_interest', 'total_

##### Create a unique index for each application

In [13]:
# create a unique number for every application as an ID column of sorts
hmda_2022_credit_score_co["unique_id"] = range(len(hmda_2022_credit_score_co))

C:\Users\Cat\AppData\Local\Temp\ipykernel_11412\2598958234.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hmda_2022_credit_score_co["unique_id"] = range(len(hmda_2022_credit_score_co))


## Converting categorical data into numeric for the purpose of running a logistic regression

### AUS Automated Underwriting System

##### Melt the AUS columns into a smaller df to OHE those columns

In [14]:
# melting data into smaller df to ohe it
hmda_melted_aus = hmda_2022_credit_score_co.melt(
    id_vars="unique_id", 
    value_vars=["aus-1", "aus-2", "aus-3", "aus-4", "aus-5"],
    var_name="aus_types", 
    value_name="aus_values"
) 

In [32]:
# inspect dataset values
hmda_melted_aus.head(2)

,unique_id,aus_types,aus_values
0,0,aus-1,1
1,1,aus-1,1


In [33]:
# inspect values in columns
hmda_melted_aus["aus_values"].unique()

array([   1,    6,    2,    3,    5,    4, 1111,    7,    0])

##### Fill in NaNs with 0s and change it to an integer not a float

In [34]:
# fill NaNs with zeroes and change the data type to integer instead of float
hmda_melted_aus["aus_values"] = hmda_melted_aus["aus_values"].fillna(0).astype(int)

In [18]:
hmda_melted_aus.head(2)

,unique_id,aus_types,aus_values
0,0,aus-1,1
1,1,aus-1,1


##### OneHotEncoder

In [19]:
# use OneHotEncoder
transformed = OneHotEncoder(sparse_output=False).set_output(transform="pandas")

# fit and transform the specific column
encoded_df = transformed.fit_transform(hmda_melted_aus[["aus_values"]])

# combine the encoded columns back with the original df
hmda_ohe = pd.concat([hmda_melted_aus.drop(columns=["aus_values"]), encoded_df], axis=1)

In [20]:
hmda_ohe.head(2) # ["aus_values_7"].unique()#.head(2)

,unique_id,aus_types,aus_values_0,aus_values_1,aus_values_2,aus_values_3,aus_values_4,aus_values_5,aus_values_6,aus_values_7,aus_values_1111
0,0,aus-1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,aus-1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [21]:
hmda_ohe.columns.unique()

Index(['unique_id', 'aus_types', 'aus_values_0', 'aus_values_1',
       'aus_values_2', 'aus_values_3', 'aus_values_4', 'aus_values_5',
       'aus_values_6', 'aus_values_7', 'aus_values_1111'],
      dtype='object')

##### Drop unnecessary columns 

In [22]:
hmda_ohe = hmda_ohe.drop(columns=["aus_values_0", "aus_types"])

##### Group by the "unique_id" column, add those values together, and reset the index

In [23]:
auses = hmda_ohe.groupby("unique_id").sum().reset_index()

In [31]:
# inspect values in columns
auses["aus_values_4"].unique()

array([0, 1, 4, 5])

In [27]:
# 
aus_columns = ['aus_values_1', 'aus_values_2', 'aus_values_3', 'aus_values_4', 'aus_values_5', 'aus_values_6','aus_values_7','aus_values_1111']

In [29]:
auses[aus_columns] = auses[aus_columns].fillna(0).astype(int)

In [30]:
auses.head(2)

,unique_id,aus_values_1,aus_values_2,aus_values_3,aus_values_4,aus_values_5,aus_values_6,aus_values_7,aus_values_1111
0,0,1,0,0,0,0,0,0,0
1,1,1,0,0,0,0,0,0,0


In [26]:
hmda_2022_credit_score_co.head(2)

,activity_year,lei,derived_msa-md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,...,denial_reason-3,denial_reason-4,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units,unique_id
1,2022,549300FGXN1K3HLB1R50,12060,GA,13113.0,1.311314e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,5777,67.49,95700,117.88,1806,2070,29,0
2,2022,549300FGXN1K3HLB1R50,12060,GA,13077.0,1.307717e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,5502,22.41,95700,141.54,1798,2180,25,1


##### Merge the AUS df and the original HMDA df together

In [27]:
hmda_aus = hmda_2022_credit_score_co.merge(auses, on="unique_id")

In [28]:
hmda_aus.head(2)

,activity_year,lei,derived_msa-md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,...,tract_median_age_of_housing_units,unique_id,aus_values_1,aus_values_2,aus_values_3,aus_values_4,aus_values_5,aus_values_6,aus_values_7,aus_values_1111
0,2022,549300FGXN1K3HLB1R50,12060,GA,13113.0,1.311314e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,29,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2022,549300FGXN1K3HLB1R50,12060,GA,13077.0,1.307717e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,25,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


##### Melt the applicant_ethnicity columns into a smaller df to OHE those columns

In [29]:
# melting data into small df to ohe it
hmda_ae_melted = hmda_aus.melt(
    id_vars="unique_id", 
    value_vars=["applicant_ethnicity-1", "applicant_ethnicity-2", "applicant_ethnicity-3", "applicant_ethnicity-4", "applicant_ethnicity-5"],
    var_name="ae_types", 
    value_name="ae_values"
)

In [30]:
hmda_ae_melted.head(2)

,unique_id,ae_types,ae_values
0,0,applicant_ethnicity-1,2.0
1,1,applicant_ethnicity-1,2.0


In [31]:
hmda_ae_melted["ae_values"].unique()

array([ 2.,  3.,  1., 14., 13., 11., 12., nan,  4.])

##### Fill in NaNs with 0s and change it to an integer not a float

In [32]:
hmda_ae_melted["ae_values"] = hmda_ae_melted["ae_values"].fillna(0).astype(int)

In [33]:
hmda_ae_melted.head(2)

,unique_id,ae_types,ae_values
0,0,applicant_ethnicity-1,2
1,1,applicant_ethnicity-1,2


##### OneHotEncode 

In [34]:
# use OneHotEncoder 
transformed = OneHotEncoder(sparse_output=False).set_output(transform="pandas")

# fit and transform the specific column
encoded_df = transformed.fit_transform(hmda_ae_melted[["ae_values"]])

# combine the encoded columns back with the original df
hmda_ae_ohe = pd.concat([hmda_ae_melted.drop(columns=["ae_values"]), encoded_df], axis=1)

In [35]:
hmda_ae_ohe.head(2)#["aus_values_7.0"].unique()#.head(2)

,unique_id,ae_types,ae_values_0,ae_values_1,ae_values_2,ae_values_3,ae_values_4,ae_values_11,ae_values_12,ae_values_13,ae_values_14
0,0,applicant_ethnicity-1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,applicant_ethnicity-1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [36]:
hmda_ae_ohe.columns.unique()

Index(['unique_id', 'ae_types', 'ae_values_0', 'ae_values_1', 'ae_values_2',
       'ae_values_3', 'ae_values_4', 'ae_values_11', 'ae_values_12',
       'ae_values_13', 'ae_values_14'],
      dtype='object')

##### Drop unnecessary columns 

In [37]:
hmda_ae_ohe = hmda_ae_ohe.drop(columns=["ae_values_0", "ae_types"])

##### Group by the "unique_id" column, add those values together, and reset the index

In [38]:
ae = hmda_ae_ohe.groupby("unique_id").sum().reset_index()#.astype(int)

In [39]:
ae["ae_values_4"].unique()

array([0., 1.])

In [40]:
ae.head(2)

,unique_id,ae_values_1,ae_values_2,ae_values_3,ae_values_4,ae_values_11,ae_values_12,ae_values_13,ae_values_14
0,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [41]:
hmda_aus.head(2)

,activity_year,lei,derived_msa-md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,...,tract_median_age_of_housing_units,unique_id,aus_values_1,aus_values_2,aus_values_3,aus_values_4,aus_values_5,aus_values_6,aus_values_7,aus_values_1111
0,2022,549300FGXN1K3HLB1R50,12060,GA,13113.0,1.311314e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,29,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2022,549300FGXN1K3HLB1R50,12060,GA,13077.0,1.307717e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,25,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


##### Merge together the Applicant Ethnicity df and the HMDA df after ohe for aus

In [42]:
hmda_ae = hmda_aus.merge(ae, on="unique_id")

In [43]:
hmda_ae.head(2)

,activity_year,lei,derived_msa-md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,...,aus_values_7,aus_values_1111,ae_values_1,ae_values_2,ae_values_3,ae_values_4,ae_values_11,ae_values_12,ae_values_13,ae_values_14
0,2022,549300FGXN1K3HLB1R50,12060,GA,13113.0,1.311314e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2022,549300FGXN1K3HLB1R50,12060,GA,13077.0,1.307717e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


##### Melt the co-applicant_ethnicity columns into a smaller df to OHE those columns

In [44]:
# melting data into small df to ohe it
hmda_cae_melted = hmda_ae.melt(
    id_vars="unique_id", 
    value_vars=["co-applicant_ethnicity-1", "co-applicant_ethnicity-2", "co-applicant_ethnicity-3", "co-applicant_ethnicity-4", "co-applicant_ethnicity-5"],
    var_name="cae_types", 
    value_name="cae_values"
)

In [45]:
hmda_cae_melted.head(2)

,unique_id,cae_types,cae_values
0,0,co-applicant_ethnicity-1,5.0
1,1,co-applicant_ethnicity-1,5.0


In [46]:
hmda_cae_melted["cae_values"].unique()

array([ 5.,  2., 13.,  3.,  1., 14., 12., 11.,  4., nan])

##### Fill in NaNs with 0s and change it to an integer not a float

In [47]:
hmda_cae_melted["cae_values"] = hmda_cae_melted["cae_values"].fillna(0).astype(int)

In [48]:
hmda_cae_melted.head(2)

,unique_id,cae_types,cae_values
0,0,co-applicant_ethnicity-1,5
1,1,co-applicant_ethnicity-1,5


##### OneHotEncode 

In [49]:
# use OneHotEncoder 
transformed = OneHotEncoder(sparse_output=False).set_output(transform="pandas")

# fit and transform the specific column
encoded_df = transformed.fit_transform(hmda_cae_melted[["cae_values"]])

# combine the encoded columns back with the original df
hmda_cae_ohe = pd.concat([hmda_cae_melted.drop(columns=["cae_values"]), encoded_df], axis=1)

In [50]:
hmda_cae_ohe.head(2)#["aus_values_7.0"].unique()#.head(2)

,unique_id,cae_types,cae_values_0,cae_values_1,cae_values_2,cae_values_3,cae_values_4,cae_values_5,cae_values_11,cae_values_12,cae_values_13,cae_values_14
0,0,co-applicant_ethnicity-1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,1,co-applicant_ethnicity-1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [51]:
hmda_cae_ohe.columns.unique()

Index(['unique_id', 'cae_types', 'cae_values_0', 'cae_values_1',
       'cae_values_2', 'cae_values_3', 'cae_values_4', 'cae_values_5',
       'cae_values_11', 'cae_values_12', 'cae_values_13', 'cae_values_14'],
      dtype='object')

##### Drop unnecessary columns 

In [52]:
hmda_cae_ohe = hmda_cae_ohe.drop(columns=["cae_values_0", "cae_types"])

##### Group by the "unique_id" column, add those values together, and reset the index

In [53]:
cae = hmda_cae_ohe.groupby("unique_id").sum().reset_index()#.astype(int)

In [54]:
cae["cae_values_4"].unique()

array([0., 1.])

In [55]:
cae.head(2)

,unique_id,cae_values_1,cae_values_2,cae_values_3,cae_values_4,cae_values_5,cae_values_11,cae_values_12,cae_values_13,cae_values_14
0,0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [56]:
hmda_ae.head(2)

,activity_year,lei,derived_msa-md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,...,aus_values_7,aus_values_1111,ae_values_1,ae_values_2,ae_values_3,ae_values_4,ae_values_11,ae_values_12,ae_values_13,ae_values_14
0,2022,549300FGXN1K3HLB1R50,12060,GA,13113.0,1.311314e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2022,549300FGXN1K3HLB1R50,12060,GA,13077.0,1.307717e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


##### Merge together the Applicant Ethnicity df and the HMDA df after ohe for aus

In [57]:
hmda_cae = hmda_ae.merge(cae, on="unique_id")

In [58]:
hmda_cae.head(2)

,activity_year,lei,derived_msa-md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,...,ae_values_14,cae_values_1,cae_values_2,cae_values_3,cae_values_4,cae_values_5,cae_values_11,cae_values_12,cae_values_13,cae_values_14
0,2022,549300FGXN1K3HLB1R50,12060,GA,13113.0,1.311314e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,2022,549300FGXN1K3HLB1R50,12060,GA,13077.0,1.307717e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


##### Melt the applicant_race columns into a smaller df to OHE those columns

In [59]:
# melting data into small df to ohe it
hmda_ar_melted = hmda_cae.melt(
    id_vars="unique_id", 
    value_vars=["applicant_race-1", "applicant_race-2", "applicant_race-3", "applicant_race-4", "applicant_race-5"],
    var_name="ar_types", 
    value_name="ar_values"
)

In [60]:
hmda_ar_melted.head(2)

,unique_id,ar_types,ar_values
0,0,applicant_race-1,5.0
1,1,applicant_race-1,5.0


In [61]:
hmda_ar_melted["ar_values"].unique()

array([ 5.,  3.,  6.,  2., 23., 25.,  1.,  4., 27., 26., 21., 24., 44.,
       22., 41., nan, 42., 43.,  7.])

In [62]:
hmda_ar_melted["ar_values"] = hmda_ar_melted["ar_values"].fillna(0).astype(int)

In [63]:
hmda_ar_melted.head(2)

,unique_id,ar_types,ar_values
0,0,applicant_race-1,5
1,1,applicant_race-1,5


##### OneHotEncode 

In [64]:
# use OneHotEncoder 
transformed = OneHotEncoder(sparse_output=False).set_output(transform="pandas")

# fit and transform the specific column
encoded_df = transformed.fit_transform(hmda_ar_melted[["ar_values"]])

# combine the encoded columns back with the original df
hmda_ar_ohe = pd.concat([hmda_ar_melted.drop(columns=["ar_values"]), encoded_df], axis=1)

In [65]:
hmda_ar_ohe.head(2)#["aus_values_7.0"].unique()#.head(2)

,unique_id,ar_types,ar_values_0,ar_values_1,ar_values_2,ar_values_3,ar_values_4,ar_values_5,ar_values_6,ar_values_7,...,ar_values_22,ar_values_23,ar_values_24,ar_values_25,ar_values_26,ar_values_27,ar_values_41,ar_values_42,ar_values_43,ar_values_44
0,0,applicant_race-1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,applicant_race-1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [66]:
hmda_ar_ohe.columns.unique()

Index(['unique_id', 'ar_types', 'ar_values_0', 'ar_values_1', 'ar_values_2',
       'ar_values_3', 'ar_values_4', 'ar_values_5', 'ar_values_6',
       'ar_values_7', 'ar_values_21', 'ar_values_22', 'ar_values_23',
       'ar_values_24', 'ar_values_25', 'ar_values_26', 'ar_values_27',
       'ar_values_41', 'ar_values_42', 'ar_values_43', 'ar_values_44'],
      dtype='object')

#### Drop unnecessary columns 

In [67]:
hmda_ar_ohe = hmda_ar_ohe.drop(columns=["ar_values_0", "ar_types"])

##### Group by the "unique_id" column, add those values together, and reset the index

In [68]:
ar = hmda_ar_ohe.groupby("unique_id").sum().reset_index()#.astype(int)

In [69]:
ar["ar_values_4"].unique()

array([0., 1.])

In [70]:
ar.head(2)

,unique_id,ar_values_1,ar_values_2,ar_values_3,ar_values_4,ar_values_5,ar_values_6,ar_values_7,ar_values_21,ar_values_22,ar_values_23,ar_values_24,ar_values_25,ar_values_26,ar_values_27,ar_values_41,ar_values_42,ar_values_43,ar_values_44
0,0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [71]:
hmda_cae.head(2)

,activity_year,lei,derived_msa-md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,...,ae_values_14,cae_values_1,cae_values_2,cae_values_3,cae_values_4,cae_values_5,cae_values_11,cae_values_12,cae_values_13,cae_values_14
0,2022,549300FGXN1K3HLB1R50,12060,GA,13113.0,1.311314e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,2022,549300FGXN1K3HLB1R50,12060,GA,13077.0,1.307717e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


##### Merge together the Applicant Ethnicity df and the HMDA df after ohe for aus

In [72]:
hmda_ar = hmda_cae.merge(ar, on="unique_id")

In [73]:
hmda_ar.head(2)

,activity_year,lei,derived_msa-md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,...,ar_values_22,ar_values_23,ar_values_24,ar_values_25,ar_values_26,ar_values_27,ar_values_41,ar_values_42,ar_values_43,ar_values_44
0,2022,549300FGXN1K3HLB1R50,12060,GA,13113.0,1.311314e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2022,549300FGXN1K3HLB1R50,12060,GA,13077.0,1.307717e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


##### Melt the co-applicant_race columns into a smaller df to OHE those columns

In [74]:
# melting data into small df to ohe it
hmda_car_melted = hmda_ar.melt(
    id_vars="unique_id", 
    value_vars=["co-applicant_race-1", "co-applicant_race-2", "co-applicant_race-3", "co-applicant_race-4", "co-applicant_race-5"],
    var_name="car_types", 
    value_name="car_values"
)

In [75]:
hmda_car_melted.head(2)

,unique_id,car_types,car_values
0,0,co-applicant_race-1,8.0
1,1,co-applicant_race-1,8.0


In [76]:
hmda_car_melted["car_values"].unique()

array([ 8.,  5., 21.,  6.,  2.,  3., 22., 27.,  1., nan, 25.,  4., 23.,
       26., 43., 24., 44., 42., 41.])

In [77]:
hmda_car_melted["car_values"] = hmda_car_melted["car_values"].fillna(0).astype(int)

In [78]:
hmda_car_melted.head(2)

,unique_id,car_types,car_values
0,0,co-applicant_race-1,8
1,1,co-applicant_race-1,8


##### OneHotEncode 

In [79]:
# use OneHotEncoder 
transformed = OneHotEncoder(sparse_output=False).set_output(transform="pandas")

# fit and transform the specific column
encoded_df = transformed.fit_transform(hmda_car_melted[["car_values"]])

# combine the encoded columns back with the original df
hmda_car_ohe = pd.concat([hmda_car_melted.drop(columns=["car_values"]), encoded_df], axis=1)

In [80]:
hmda_car_ohe.head(2)

,unique_id,car_types,car_values_0,car_values_1,car_values_2,car_values_3,car_values_4,car_values_5,car_values_6,car_values_8,...,car_values_22,car_values_23,car_values_24,car_values_25,car_values_26,car_values_27,car_values_41,car_values_42,car_values_43,car_values_44
0,0,co-applicant_race-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,co-applicant_race-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [81]:
hmda_car_ohe.columns.unique()

Index(['unique_id', 'car_types', 'car_values_0', 'car_values_1',
       'car_values_2', 'car_values_3', 'car_values_4', 'car_values_5',
       'car_values_6', 'car_values_8', 'car_values_21', 'car_values_22',
       'car_values_23', 'car_values_24', 'car_values_25', 'car_values_26',
       'car_values_27', 'car_values_41', 'car_values_42', 'car_values_43',
       'car_values_44'],
      dtype='object')

##### Drop unnecessary columns 

In [82]:
hmda_car_ohe = hmda_car_ohe.drop(columns=["car_values_0", "car_types"])

##### Group by the "unique_id" column, add those values together, and reset the index

In [83]:
car = hmda_car_ohe.groupby("unique_id").sum().reset_index()#.astype(int)

In [84]:
car["car_values_4"].unique()

array([0., 1.])

In [85]:
car.head(2)

,unique_id,car_values_1,car_values_2,car_values_3,car_values_4,car_values_5,car_values_6,car_values_8,car_values_21,car_values_22,car_values_23,car_values_24,car_values_25,car_values_26,car_values_27,car_values_41,car_values_42,car_values_43,car_values_44
0,0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [86]:
hmda_ar.head(2)

,activity_year,lei,derived_msa-md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,...,ar_values_22,ar_values_23,ar_values_24,ar_values_25,ar_values_26,ar_values_27,ar_values_41,ar_values_42,ar_values_43,ar_values_44
0,2022,549300FGXN1K3HLB1R50,12060,GA,13113.0,1.311314e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2022,549300FGXN1K3HLB1R50,12060,GA,13077.0,1.307717e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


##### Merge together the Applicant Ethnicity df and the HMDA df after ohe for aus

In [87]:
hmda_car = hmda_ar.merge(car, on="unique_id")

In [88]:
hmda_car.head(2)

,activity_year,lei,derived_msa-md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,...,car_values_22,car_values_23,car_values_24,car_values_25,car_values_26,car_values_27,car_values_41,car_values_42,car_values_43,car_values_44
0,2022,549300FGXN1K3HLB1R50,12060,GA,13113.0,1.311314e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2022,549300FGXN1K3HLB1R50,12060,GA,13077.0,1.307717e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


##### Melt the denial_reason columns into a smaller df to OHE those columns


In [89]:
# melting data into small df to ohe it
hmda_dr_melted = hmda_car.melt(
    id_vars="unique_id", 
    value_vars=["denial_reason-1", "denial_reason-2", "denial_reason-3", "denial_reason-4"],
    var_name="dr_types", 
    value_name="dr_values"
)

In [90]:
hmda_dr_melted.head(2)

,unique_id,dr_types,dr_values
0,0,denial_reason-1,10.0
1,1,denial_reason-1,10.0


In [91]:
hmda_dr_melted["dr_values"].unique()

array([1.000e+01, 1.000e+00, 7.000e+00, 6.000e+00, 4.000e+00, 9.000e+00,
       5.000e+00, 3.000e+00, 2.000e+00, 8.000e+00, 1.111e+03,       nan])

In [92]:
hmda_dr_melted["dr_values"] = hmda_dr_melted["dr_values"].fillna(0).astype(int)

In [93]:
hmda_dr_melted.head(2)

,unique_id,dr_types,dr_values
0,0,denial_reason-1,10
1,1,denial_reason-1,10


##### OneHotEncode 

In [94]:
# use OneHotEncoder 
transformed = OneHotEncoder(sparse_output=False).set_output(transform="pandas")

# fit and transform the specific column
encoded_df = transformed.fit_transform(hmda_dr_melted[["dr_values"]])

# combine the encoded columns back with the original df
hmda_dr_ohe = pd.concat([hmda_dr_melted.drop(columns=["dr_values"]), encoded_df], axis=1)

In [95]:
hmda_dr_ohe.head(2)

,unique_id,dr_types,dr_values_0,dr_values_1,dr_values_2,dr_values_3,dr_values_4,dr_values_5,dr_values_6,dr_values_7,dr_values_8,dr_values_9,dr_values_10,dr_values_1111
0,0,denial_reason-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1,denial_reason-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [96]:
hmda_dr_ohe.columns.unique()

Index(['unique_id', 'dr_types', 'dr_values_0', 'dr_values_1', 'dr_values_2',
       'dr_values_3', 'dr_values_4', 'dr_values_5', 'dr_values_6',
       'dr_values_7', 'dr_values_8', 'dr_values_9', 'dr_values_10',
       'dr_values_1111'],
      dtype='object')

##### Drop unnecessary columns 

In [97]:
hmda_dr_ohe = hmda_dr_ohe.drop(columns=["dr_values_0", "dr_types"])

##### Group by the "unique_id" column, add those values together, and reset the index

In [98]:
dr = hmda_dr_ohe.groupby("unique_id").sum().reset_index()#.astype(int)

In [99]:
dr["dr_values_4"].unique()

array([0., 1.])

In [100]:
dr.head(2)

,unique_id,dr_values_1,dr_values_2,dr_values_3,dr_values_4,dr_values_5,dr_values_6,dr_values_7,dr_values_8,dr_values_9,dr_values_10,dr_values_1111
0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [101]:
hmda_car.head(2)

,activity_year,lei,derived_msa-md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,...,car_values_22,car_values_23,car_values_24,car_values_25,car_values_26,car_values_27,car_values_41,car_values_42,car_values_43,car_values_44
0,2022,549300FGXN1K3HLB1R50,12060,GA,13113.0,1.311314e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2022,549300FGXN1K3HLB1R50,12060,GA,13077.0,1.307717e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


##### Merge together the Applicant Ethnicity df and the HMDA df after ohe for denial reason

In [102]:
hmda_ohe_all = hmda_car.merge(dr, on="unique_id")

In [103]:
hmda_ohe_all.head(2)

,activity_year,lei,derived_msa-md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,...,dr_values_2,dr_values_3,dr_values_4,dr_values_5,dr_values_6,dr_values_7,dr_values_8,dr_values_9,dr_values_10,dr_values_1111
0,2022,549300FGXN1K3HLB1R50,12060,GA,13113.0,1.311314e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,2022,549300FGXN1K3HLB1R50,12060,GA,13077.0,1.307717e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


##### Convert the loan_to_income_ratio column to binaries OHE those columns

In [104]:
# melting data into small df to ohe it
# hmda_lir_melted = hmda_ohe_all.melt(
#     id_vars="unique_id", 
#     value_vars=["denial_reason-1", "denial_reason-2", "denial_reason-3", "denial_reason-4"],
#     var_name="dr_types", 
#     value_name="dr_values"
# )

In [105]:
# hmda_dr_melted.head(2)

In [106]:
# hmda_dr_melted["dr_values"].unique()

In [107]:
# hmda_dr_melted["dr_values"] = hmda_dr_melted["dr_values"].fillna(0).astype(int)

In [108]:
# hmda_dr_melted.head(2)

In [109]:
hmda_ohe_all["debt_to_income_ratio"].unique()

array(['38', '20%-<30%', '30%-<36%', '<20%', nan, '48', '36', '39',
       '50%-60%', '41', '37', '47', '46', '44', '42', '49', '43', '40',
       '45', '>60%'], dtype=object)

##### OneHotEncode 

In [110]:
# use OneHotEncoder, same as before
transformed = OneHotEncoder(sparse_output=False).set_output(transform="pandas")

# fit and transform the specific column
encoded_ltir = transformed.fit_transform(hmda_ohe_all[["debt_to_income_ratio"]])

# combine the encoded columns back with the original df
ltir_model = pd.concat([hmda_ohe_all.drop(columns=["debt_to_income_ratio"]), encoded_ltir], axis=1)

In [111]:
ltir_model.columns.tolist()

['activity_year',
 'lei',
 'derived_msa-md',
 'state_code',
 'county_code',
 'census_tract',
 'conforming_loan_limit',
 'derived_loan_product_type',
 'derived_dwelling_category',
 'derived_ethnicity',
 'derived_race',
 'derived_sex',
 'action_taken',
 'purchaser_type',
 'preapproval',
 'loan_type',
 'loan_purpose',
 'lien_status',
 'reverse_mortgage',
 'open-end_line_of_credit',
 'business_or_commercial_purpose',
 'loan_amount',
 'loan_to_value_ratio',
 'interest_rate',
 'rate_spread',
 'hoepa_status',
 'total_loan_costs',
 'total_points_and_fees',
 'origination_charges',
 'discount_points',
 'lender_credits',
 'loan_term',
 'prepayment_penalty_term',
 'intro_rate_period',
 'negative_amortization',
 'interest_only_payment',
 'balloon_payment',
 'other_nonamortizing_features',
 'property_value',
 'construction_method',
 'occupancy_type',
 'manufactured_home_secured_property_type',
 'manufactured_home_land_property_interest',
 'total_units',
 'multifamily_affordable_units',
 'income',
 '

In [112]:
ltir_model = ltir_model.rename(columns = {'debt_to_income_ratio_20%-<30%':'debt_to_income_ratio_20_to_less_than_30_percent', 'debt_to_income_ratio_30%-<36%':'debt_to_income_ratio_30_to_less_than_36_percent', 'debt_to_income_ratio_50%-60%':'debt_to_income_ratio_50_to_60_percent', 'debt_to_income_ratio_<20%':'debt_to_income_ratio_less_than_20_percent', 'debt_to_income_ratio_>60%':'debt_to_income_ratio_greater_than_60_percent'}).head(2)

In [113]:
ltir_model.columns.tolist()

['activity_year',
 'lei',
 'derived_msa-md',
 'state_code',
 'county_code',
 'census_tract',
 'conforming_loan_limit',
 'derived_loan_product_type',
 'derived_dwelling_category',
 'derived_ethnicity',
 'derived_race',
 'derived_sex',
 'action_taken',
 'purchaser_type',
 'preapproval',
 'loan_type',
 'loan_purpose',
 'lien_status',
 'reverse_mortgage',
 'open-end_line_of_credit',
 'business_or_commercial_purpose',
 'loan_amount',
 'loan_to_value_ratio',
 'interest_rate',
 'rate_spread',
 'hoepa_status',
 'total_loan_costs',
 'total_points_and_fees',
 'origination_charges',
 'discount_points',
 'lender_credits',
 'loan_term',
 'prepayment_penalty_term',
 'intro_rate_period',
 'negative_amortization',
 'interest_only_payment',
 'balloon_payment',
 'other_nonamortizing_features',
 'property_value',
 'construction_method',
 'occupancy_type',
 'manufactured_home_secured_property_type',
 'manufactured_home_land_property_interest',
 'total_units',
 'multifamily_affordable_units',
 'income',
 '

In [114]:
ltir_model['debt_to_income_ratio_greater_than_60_percent']

0    0.0
1    0.0
Name: debt_to_income_ratio_greater_than_60_percent, dtype: float64

In [115]:
dtir = ['debt_to_income_ratio_20_to_less_than_30_percent', 'ae_values_2', 'ae_values_11', 'ae_values_12', 'ae_values_13', 'ae_values_14']

In [116]:
ltir_model[dtir] = ltir_model[dtir].astype(int)

In [117]:
hmda_dr_ohe.columns.unique()

Index(['unique_id', 'dr_values_1', 'dr_values_2', 'dr_values_3', 'dr_values_4',
       'dr_values_5', 'dr_values_6', 'dr_values_7', 'dr_values_8',
       'dr_values_9', 'dr_values_10', 'dr_values_1111'],
      dtype='object')

##### Drop unnecessary columns 

In [118]:
# hmda_dr_ohe = hmda_dr_ohe.drop(columns=["dr_values_0", "dr_types"])

##### Group by the "unique_id" column, add those values together, and reset the index

In [119]:
dr = hmda_dr_ohe.groupby("unique_id").sum().reset_index()#.astype(int)

In [120]:
dr["dr_values_4"].unique()

array([0., 1.])

In [121]:
dr.head(2)

,unique_id,dr_values_1,dr_values_2,dr_values_3,dr_values_4,dr_values_5,dr_values_6,dr_values_7,dr_values_8,dr_values_9,dr_values_10,dr_values_1111
0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [122]:
hmda_car.head(2)

,activity_year,lei,derived_msa-md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,...,car_values_22,car_values_23,car_values_24,car_values_25,car_values_26,car_values_27,car_values_41,car_values_42,car_values_43,car_values_44
0,2022,549300FGXN1K3HLB1R50,12060,GA,13113.0,1.311314e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2022,549300FGXN1K3HLB1R50,12060,GA,13077.0,1.307717e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


##### Merge together the Applicant Ethnicity df and the HMDA df after ohe for denial reasons

In [123]:
hmda_ohe_all = hmda_car.merge(dr, on="unique_id")

In [124]:
hmda_2023.columns

Index(['activity_year', 'lei', 'derived_msa-md', 'state_code', 'county_code',
       'census_tract', 'conforming_loan_limit', 'derived_loan_product_type',
       'derived_dwelling_category', 'derived_ethnicity', 'derived_race',
       'derived_sex', 'action_taken', 'purchaser_type', 'preapproval',
       'loan_type', 'loan_purpose', 'lien_status', 'reverse_mortgage',
       'open-end_line_of_credit', 'business_or_commercial_purpose',
       'loan_amount', 'loan_to_value_ratio', 'interest_rate', 'rate_spread',
       'hoepa_status', 'total_loan_costs', 'total_points_and_fees',
       'origination_charges', 'discount_points', 'lender_credits', 'loan_term',
       'prepayment_penalty_term', 'intro_rate_period', 'negative_amortization',
       'interest_only_payment', 'balloon_payment',
       'other_nonamortizing_features', 'property_value', 'construction_method',
       'occupancy_type', 'manufactured_home_secured_property_type',
       'manufactured_home_land_property_interest', 'total_

In [125]:
hmda_ohe_all["applicant_ethnicity-1"]

0         2.0
1         2.0
2         2.0
3         2.0
4         2.0
         ... 
181006    2.0
181007    2.0
181008    3.0
181009    1.0
181010    2.0
Name: applicant_ethnicity-1, Length: 181011, dtype: float64

In [126]:
hmda_ohe_all["action_taken"].unique()

array([1, 2, 3])

In [127]:
# smf.logit("action_taken ~ ae_values_1", data = hmda_ohe_all). fit()

## Is this doing what I think or want it to do MH?

In [128]:
# want to keep approved/originated (1) and application denied (3) options only
list_of_values_to_keep = [1,3]

# filter to only those options in the column
hmda_ohe_all_at = hmda_ohe_all[hmda_ohe_all['action_taken'].isin(list_of_values_to_keep)]

# make a approved_originated column and make it a binary for whether something was approved/originated or denied
hmda_ohe_all_at['approved_originated'] = (hmda_ohe_all_at['action_taken'] == 1).astype(int)

C:\Users\Cat\AppData\Local\Temp\ipykernel_12924\2010664715.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hmda_ohe_all_at['approved_originated'] = (hmda_ohe_all_at['action_taken'] == 1).astype(int)


In [129]:
hmda_ohe_all_at = hmda_ohe_all_at[(hmda_ohe_all_at['ae_values_3'] == 0) & (hmda_ohe_all_at['ae_values_4'] == 0)]

In [130]:
hmda_ohe_all_at['action_taken'].unique()

array([1, 3])

In [131]:
# I don't want to include ae_values_3 and ae_values_4 because they are not provided or not applicable

In [132]:
# make my list of applicant ethnicity columns minus the White/"Not Hispanic or Latino" column (ae_value_2)
ethnicities = ['ae_values_1', 'ae_values_11', 'ae_values_12', 'ae_values_13', 'ae_values_14']

In [133]:
# make my list of contorl variable columns
controls = ['income', 'debt_to_income_ratio']

In [134]:
# build the table/model for logistic regression
model = hmda_ohe_all_at[['approved_originated'] + ethnicities + controls]

In [135]:
# change the ethnicities from floats to ints
model[ethnicities] = model[ethnicities].astype(int)

C:\Users\Cat\AppData\Local\Temp\ipykernel_12924\1797546527.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  model[ethnicities] = model[ethnicities].astype(int)


### Need to figure out what to do with these variables that fall outside of the binary and why are there so many zeroes

In [136]:
# checking if some rows have more than one ethnicity listed
model[ethnicities].sum(axis=1).value_counts()

0    127264
1      7085
2      6150
3        94
4         7
5         3
Name: count, dtype: int64

### checked to see if the non-hispanic column was to blame for all the zeroes and it appears to be the culprit

In [137]:
# make my list of applicant ethnicity columns including the White/"Not Hispanic or Latino" column (ae_value_2)
exp_ethnicities = ['ae_values_1', 'ae_values_2', 'ae_values_11', 'ae_values_12', 'ae_values_13', 'ae_values_14']

In [138]:
# build the table/model for logistic regression
exp_model = hmda_ohe_all_at[['approved_originated'] + exp_ethnicities + controls]

In [139]:
# change the ethnicities from floats to ints
exp_model[exp_ethnicities] = exp_model[exp_ethnicities].astype(int)

C:\Users\Cat\AppData\Local\Temp\ipykernel_12924\3099246249.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  exp_model[exp_ethnicities] = exp_model[exp_ethnicities].astype(int)


In [140]:
# checking if some rows have more than one ethnicity listed and reason for so many zeroes
exp_model[exp_ethnicities].sum(axis=1).value_counts()

1    133581
2      6763
0       129
3       119
4         7
5         4
Name: count, dtype: int64